In [1]:
import numpy as np
import networkx as nx
from collections import defaultdict
from typing import Dict, List, Set, Tuple

class GRNEmbbeddingCycleDetector:
    def __init__(self, dataset_id: int, embeddings: np.ndarray, threshold: float = 0.5):
        """Initialize cycle detector using embeddings.
        
        Args:
            dataset_id: ID of the dataset
            embeddings: Gene embeddings matrix of shape (num_genes, embedding_dim)
            threshold: Similarity threshold for edge prediction
        """
        self.dataset_id = dataset_id
        self.embeddings = embeddings
        self.threshold = threshold
        self.num_genes = embeddings.shape[0]
        self.graph = self._build_graph_from_embeddings()
        
    def _build_graph_from_embeddings(self) -> Dict[int, List[int]]:
        """Convert embeddings to adjacency list representation."""
        # Compute similarity matrix using dot product
        similarity_matrix = np.dot(self.embeddings, self.embeddings.T)
        
        # Normalize similarities to [0,1]
        similarity_matrix = (similarity_matrix + 1) / 2
        
        # Convert to adjacency list
        graph = defaultdict(list)
        for i in range(self.num_genes):
            for j in range(self.num_genes):
                if i != j and similarity_matrix[i,j] > self.threshold:
                    graph[i].append(j)
                    
        return graph
        
    def _build_networkx_graph(self) -> nx.DiGraph:
        """Convert adjacency list to NetworkX directed graph."""
        G = nx.DiGraph()
        for node in self.graph:
            for neighbor in self.graph[node]:
                G.add_edge(node, neighbor)
        return G
    
    def find_cycles(self) -> List[List[int]]:
        """Find all cycles in the network using NetworkX."""
        G = self._build_networkx_graph()
        try:
            cycles = list(nx.simple_cycles(G))
            return cycles
        except Exception as e:
            print(f"Error finding cycles: {e}")
            return []
            
    def get_cycle_stats(self) -> Dict:
        """Get comprehensive cycle statistics."""
        cycles = self.find_cycles()
        G = self._build_networkx_graph()
        
        return {
            'total_cycles': len(cycles),
            'cycles': cycles,
            'genes_in_cycles': len(set(node for cycle in cycles for node in cycle)),
            'cycle_lengths': [len(cycle) for cycle in cycles],
            'strongly_connected_components': list(nx.strongly_connected_components(G)),
            'avg_cycle_length': np.mean([len(cycle) for cycle in cycles]) if cycles else 0,
            'max_cycle_length': max([len(cycle) for cycle in cycles]) if cycles else 0,
            'min_cycle_length': min([len(cycle) for cycle in cycles]) if cycles else 0
        }
    
    def visualize_cycles(self, save_path: str = None):
        """Visualize detected cycles using NetworkX."""
        import matplotlib.pyplot as plt
        
        G = self._build_networkx_graph()
        cycles = self.find_cycles()
        
        plt.figure(figsize=(12, 8))
        pos = nx.spring_layout(G)
        
        # Draw all edges in light gray
        nx.draw_networkx_edges(G, pos, edge_color='lightgray', arrows=True)
        
        # Draw cycle edges in different colors
        colors = plt.cm.rainbow(np.linspace(0, 1, len(cycles)))
        for cycle, color in zip(cycles, colors):
            cycle_edges = list(zip(cycle, cycle[1:] + [cycle[0]]))
            nx.draw_networkx_edges(G, pos, edgelist=cycle_edges, 
                                 edge_color=[color], arrows=True, width=2)
        
        # Draw nodes
        nx.draw_networkx_nodes(G, pos, node_color='lightblue')
        nx.draw_networkx_labels(G, pos)
        
        plt.title(f"Gene Regulatory Network Cycles (Dataset {self.dataset_id})")
        if save_path:
            plt.savefig(save_path)
        plt.close()


In [9]:
import os
import re
def get_datasets():
    datasets = []
    data_sets_dir = '../SERGIO/data_sets'
    for folder_name in os.listdir(data_sets_dir):
        dataset_info = parse_dataset_name(folder_name)
        if dataset_info:
            datasets.append(dataset_info)
    # Include new datasets
    new_datasets = [
        {
            'dataset_id': 1001,
            'dataset_name': 'mESC',
            'expression_file': 'data/raws/mESC-ExpressionData.csv',
            'network_file': 'data/raws/mESC-network.csv',
        },
        {
            'dataset_id': 1002,
            'dataset_name': 'mHSC-E',
            'expression_file': 'data/raws/mHSC-E-ExpressionData.csv',
            'network_file': 'data/raws/mHSC-E-network.csv',
        },
        {
            'dataset_id': 1003,
            'dataset_name': 'mHSC-GM',
            'expression_file': 'data/raws/mHSC-GM-ExpressionData.csv',
            'network_file': 'data/raws/mHSC-GM-network.csv',
        },
        {
            'dataset_id': 1004,
            'dataset_name': 'mHSC-L',
            'expression_file': 'data/raws/mHSC-L-ExpressionData.csv',
            'network_file': 'data/raws/mHSC-L-network.csv',
        },
        {
            'dataset_id': 1005,
            'dataset_name': 'mESC-200',
            'expression_file': 'data/raws/mESC-200-ExpressionData.csv',
            'network_file': 'data/raws/mESC-200-network.csv',
        },
        {
            'dataset_id': 1006,
            'dataset_name': 'mHSC-E-200',
            'expression_file': 'data/raws/mHSC-E-200-ExpressionData.csv',
            'network_file': 'data/raws/mHSC-E-200-network.csv',
        },
    ]
    datasets.extend(new_datasets)
    return datasets

def parse_dataset_name(folder_name):
    pattern1 = r'De-noised_(\d+)G_(\d+)T_(\d+)cPerT_dynamics_(\d+)_DS(\d+)'
    pattern2 = r'De-noised_(\d+)G_(\d+)T_(\d+)cPerT_(\d+)_DS(\d+)'
    match_p1 = re.match(pattern1, folder_name)
    match_p2 = re.match(pattern2, folder_name)
    if match_p1:
        return {
            'number_genes': int(match_p1.group(1)),
            'number_bins': int(match_p1.group(2)),
            'cells_per_type': int(match_p1.group(3)),
            'dynamics': int(match_p1.group(4)),
            'dataset_id': int(match_p1.group(5)),
            'folder_name': folder_name
        }
    if match_p2:
        return {
            'number_genes': int(match_p2.group(1)),
            'number_bins': int(match_p2.group(2)),
            'cells_per_type': int(match_p2.group(3)),
            'dynamics': int(match_p2.group(4)),
            'dataset_id': int(match_p2.group(5)),
            'folder_name': folder_name
        }
    return


In [10]:
datasets = get_datasets()
for dataset_info in datasets[7:]:
    dataset_id = dataset_info['dataset_id']
    embs = np.load(f'./results/cl/DS{dataset_id}/vae_embeddings.npy')
    
    # Create detector
    detector = GRNEmbbeddingCycleDetector(dataset_id, embs)
    
    # Get cycle statistics
    stats = detector.get_cycle_stats()
    
    print(f"\nDataset {dataset_id} Analysis:")
    print(f"Total cycles: {stats['total_cycles']}")
    print(f"Average cycle length: {stats['avg_cycle_length']:.2f}")
    print(f"Number of genes involved in cycles: {stats['genes_in_cycles']}")
    
    # Visualize cycles
    detector.visualize_cycles(f'cycles_DS{dataset_id}.png')
    
    # Print first few cycles if any exist
    if stats['cycles']:
        print("\nExample cycles:")
        for i, cycle in enumerate(stats['cycles'][:3], 1):
            print(f"Cycle {i}: {' -> '.join(map(str, cycle + [cycle[0]]))}")

: 

: 

: 